In [2]:
import sqlite3
import pandas as pd

conn= sqlite3.connect('flightdelay_project.db')

### NCEI NOAA Weather Dataset
This dataset contains daily weather records for 20 major U.S. airports from January to April 2025.
Source: NOAA National Centers for Environmental Information (NCEI).
- Loading dataset into SQLite database:

In [3]:
weather_df= pd.read_csv ( 
     r"C:\Users\what1\OneDrive\Documents\Flight Delay Prediction Model\data\raw\weatherdata\weather_2025_jan_apr.csv"
)

weather_df.to_sql(
    "weather_raw",
    conn, if_exists="replace", index=False
)

2400

### Weather Data Structure Check
This section checks the row count, airport coverage, date range, and required columns in the raw NOAA weather table.

In [4]:
wq1_rowcount= """
SELECT COUNT(*) as ROW_COUNT FROM weather_raw;"""

pd.read_sql_query (wq1_rowcount, conn) 

,ROW_COUNT
0,2400


In [5]:
wq2_tableinfo= """
PRAGMA table_info (weather_raw);"""


pd.read_sql_query (wq2_tableinfo, conn) 

,cid,name,type,notnull,dflt_value,pk
0,0,STATION,INTEGER,0,None,0
1,1,DATE,TEXT,0,None,0
2,2,LATITUDE,REAL,0,None,0
3,3,LONGITUDE,REAL,0,None,0
4,4,ELEVATION,REAL,0,None,0
5,5,NAME,TEXT,0,None,0
6,6,TEMP,REAL,0,None,0
7,7,TEMP_ATTRIBUTES,INTEGER,0,None,0
8,8,DEWP,REAL,0,None,0
9,9,DEWP_ATTRIBUTES,INTEGER,0,None,0


In [29]:
wq3_datecheck= """
SELECT 
 MIN (DATE) AS START_DATE, 
 MAX (DATE) AS END_DATE
FROM weather_raw;
"""

pd.read_sql_query (wq3_datecheck, conn) 

,START_DATE,END_DATE
0,2025-01-01,2025-04-30


In [7]:
wq4_airportcheck= """
SELECT 
 AIRPORT, COUNT (*) 
FROM weather_raw
GROUP BY AIRPORT
ORDER BY AIRPORT;
"""

pd.read_sql_query(wq4_airportcheck, conn)

,AIRPORT,COUNT (*)
0,ATL,120
1,BOS,120
2,CLT,120
3,DCA,120
4,DEN,120
5,DFW,120
6,DTW,120
7,EWR,120
8,IAH,120
9,LAS,120


In [8]:
wq5_monthcheck="""
SELECT 
 substr(DATE,1,7) AS YEAR_MONTH,
 COUNT(*) AS ROW_COUNT
 FROM weather_raw
GROUP BY substr(DATE,1,7)
ORDER BY YEAR_MONTH; 
"""

pd.read_sql_query(wq5_monthcheck, conn)

,YEAR_MONTH,ROW_COUNT
0,2025-01,620
1,2025-02,560
2,2025-03,620
3,2025-04,600


In [9]:
wq6_columns= """
SELECT 
 AIRPORT,
 DATE, 
 TEMP,
 VISIB, 
 WDSP,
 PRCP,
 FRSHTT
FROM weather_raw
LIMIT 10; """


pd.read_sql_query (wq6_columns, conn) 

,AIRPORT,DATE,TEMP,VISIB,WDSP,PRCP,FRSHTT
0,DFW,2025-01-01,44.7,10.0,6.8,0.00,0
1,DFW,2025-01-02,45.7,10.0,4.4,0.00,0
2,DFW,2025-01-03,50.5,9.9,4.4,0.00,0
3,DFW,2025-01-04,58.9,10.0,9.2,0.00,0
4,DFW,2025-01-05,55.8,9.8,18.6,0.00,10000
5,DFW,2025-01-06,28.7,10.0,16.9,0.00,0
6,DFW,2025-01-07,31.0,10.0,8.8,0.00,0
7,DFW,2025-01-08,32.9,10.0,9.3,0.00,0
8,DFW,2025-01-09,33.6,6.7,3.8,0.00,11000
9,DFW,2025-01-10,33.3,4.6,10.2,1.56,11000


In [17]:
wq7_duplicates="""
SELECT 
 AIRPORT,
 DATE,
 COUNT(*) AS DUPLICATE_COUNT
FROM weather_raw 
GROUP BY
 AIRPORT,
 DATE
HAVING COUNT(*) >1
ORDER BY DUPLICATE_COUNT DESC
LIMIT 10;
"""
pd.read_sql_query (wq7_duplicates,conn)

,AIRPORT,DATE,DUPLICATE_COUNT


- Date ranges, airport count (20 selected) are clean with no recorded duplicates. 
-  FRSHTT - Indicators per NOAA NCEI data:
                         Fog ('F' - 1st digit).
                         Rain or Drizzle ('R' - 2nd digit).
                         Snow or Ice Pellets ('S' - 3rd digit).
                         Hail ('H' - 4th digit).
                         Thunder ('T' - 5th digit).
                         Tornado or Funnel Cloud ('T' - 6th digit)
- FRSHTT column recorded as integers & will need to be converted to 06 decimal points during further cleaning to support indication. 


### Missing and Placeholder Value Checks
NOAA GSOD uses sentinel values such as 9999.9, 999.9, and 99.99 to represent missing measurements. These checks confirm whether those values appear in the weather columns needed for modeling. 

In [18]:
wq8_missing_blanks= """
SELECT 
 SUM (CASE WHEN DATE IS NULL THEN 1 ELSE 0 END) AS DATE_MISSING, 
 SUM (CASE WHEN TRIM(DATE)='' THEN 1 ELSE 0 END) AS DATE_BLANK, 
 SUM (CASE WHEN AIRPORT IS NULL THEN 1 ELSE 0 END) AS AIRPORT_MISSING,
 SUM (CASE WHEN TRIM(AIRPORT)='' THEN 1 ELSE 0 END) AS AIRPORT_BLANK,
 SUM (CASE WHEN TEMP IS NULL THEN 1 ELSE 0 END) AS TEMP_MISSING,
 SUM (CASE WHEN PRCP IS NULL THEN 1 ELSE 0 END) AS PRCP_MISSING, 
 SUM (CASE WHEN WDSP IS NULL THEN 1 ELSE 0 END) AS WDSP_MISSING,
 SUM (CASE WHEN VISIB IS NULL THEN 1 ELSE 0 END) AS VISIB_MISSING, 
 SUM (CASE WHEN FRSHTT IS NULL THEN 1 ELSE 0 END) AS FRSHTT_MISSING
FROM weather_raw; 
"""
pd.read_sql_query(wq8_missing_blanks, conn)

,DATE_MISSING,DATE_BLANK,AIRPORT_MISSING,AIRPORT_BLANK,TEMP_MISSING,PRCP_MISSING,WDSP_MISSING,VISIB_MISSING,FRSHTT_MISSING
0,0,0,0,0,0,0,0,0,0


In [19]:
wq9_sentinel_check="""
SELECT 
 SUM (CASE WHEN TEMP = 9999.9 THEN 1 ELSE 0 END) AS TEMP_PLACEHOLDER, 
 SUM (CASE WHEN PRCP = 99.99 THEN 1 ELSE 0 END) AS PRCP_PLACEHOLDER, 
 SUM (CASE WHEN VISIB = 999.9 THEN 1 ELSE 0 END) AS VISIB_PLACEHOLDER, 
 SUM (CASE WHEN WDSP = 999.9 THEN 1 ELSE 0 END) AS WDSP_PLACEHOLDER, 
 SUM (CASE WHEN FRSHTT <0 THEN 1 ELSE 0 END) AS FRSHTT_PLACEHOLDER
FROM weather_raw;
"""
pd.read_sql_query(wq9_sentinel_check, conn)

,TEMP_PLACEHOLDER,PRCP_PLACEHOLDER,VISIB_PLACEHOLDER,WDSP_PLACEHOLDER,FRSHTT_PLACEHOLDER
0,0,0,0,0,0


Preview of rows to keep:

In [21]:
wq10_preview= """
SELECT 
 DATE, 
 AIRPORT, 
 TEMP,
 VISIB,
 PRCP,
 WDSP,
 FRSHTT
FROM weather_raw
LIMIT 10; 
"""

pd.read_sql_query(wq10_preview, conn) 

,DATE,AIRPORT,TEMP,VISIB,PRCP,WDSP,FRSHTT
0,2025-01-01,DFW,44.7,10.0,0.00,6.8,0
1,2025-01-02,DFW,45.7,10.0,0.00,4.4,0
2,2025-01-03,DFW,50.5,9.9,0.00,4.4,0
3,2025-01-04,DFW,58.9,10.0,0.00,9.2,0
4,2025-01-05,DFW,55.8,9.8,0.00,18.6,10000
5,2025-01-06,DFW,28.7,10.0,0.00,16.9,0
6,2025-01-07,DFW,31.0,10.0,0.00,8.8,0
7,2025-01-08,DFW,32.9,10.0,0.00,9.3,0
8,2025-01-09,DFW,33.6,6.7,0.00,3.8,11000
9,2025-01-10,DFW,33.3,4.6,1.56,10.2,11000


### Create Cleaned Weather Table

This step creates `weather_cleaned` from the raw NOAA weather table. Only the weather columns needed for the flight delay model are kept: date, airport, temperature, visibility, wind speed, precipitation, and weather condition flags (FRSHTT).

NOAA sentinel values are converted to `NULL` so they are treated as missing values instead of real weather measurements which will later be imputed in python for further cleaning & EDA. The table stays at one row per airport per date, which will later be merged with `flights_cleaned` using `AIRPORT + DATE`.

In [22]:
wq11_create_weather_cleaned= """
DROP TABLE IF EXISTS weather_cleaned; 

CREATE TABLE weather_cleaned AS
SELECT 
 DATE AS WEATHER_DATE, 
 UPPER(TRIM(AIRPORT)) AS AIRPORT, 
 CASE WHEN TEMP=9999.9 THEN NULL ELSE CAST(TEMP AS REAL) END AS TEMPERATURE, 
 CASE WHEN VISIB=999.9 THEN NULL ELSE CAST(VISIB AS REAL) END AS VISIBILITY, 
 CASE WHEN WDSP=999.9 THEN NULL ELSE CAST(WDSP AS REAL) END AS WINDSPEED, 
 CASE WHEN PRCP=99.99 THEN NULL ELSE CAST(PRCP AS REAL) END AS PRECIPITATION,

 printf('%06d', FRSHTT) AS WEATHER_FLAGS

FROM weather_raw
WHERE DATE BETWEEN '2025-01-01' AND '2025-04-30' 
 AND DATE IS NOT NULL
 AND TRIM(DATE)<>''
 AND AIRPORT IS NOT NULL
 AND TRIM(AIRPORT)<>'';
"""

conn.executescript(wq11_create_weather_cleaned)
conn.commit() 

In [32]:
wq12_preview_weather_cleaned="""
SELECT * FROM weather_cleaned
LIMIT 20;
"""

pd.read_sql_query(wq12_preview_weather_cleaned,conn)

,WEATHER_DATE,AIRPORT,TEMPERATURE,VISIBILITY,WINDSPEED,PRECIPITATION,WEATHER_FLAGS
0,2025-01-01,DFW,44.7,10.0,6.8,0.00,000000
1,2025-01-02,DFW,45.7,10.0,4.4,0.00,000000
2,2025-01-03,DFW,50.5,9.9,4.4,0.00,000000
3,2025-01-04,DFW,58.9,10.0,9.2,0.00,000000
4,2025-01-05,DFW,55.8,9.8,18.6,0.00,010000
5,2025-01-06,DFW,28.7,10.0,16.9,0.00,000000
6,2025-01-07,DFW,31.0,10.0,8.8,0.00,000000
7,2025-01-08,DFW,32.9,10.0,9.3,0.00,000000
8,2025-01-09,DFW,33.6,6.7,3.8,0.00,011000
9,2025-01-10,DFW,33.3,4.6,10.2,1.56,011000


Final Data Health Check:

In [31]:
wq13_checkrows_cleaned="""
SELECT 
 COUNT(*) AS ROW_COUNT, 
 COUNT(DISTINCT AIRPORT) AS AIRPORT_COUNT, 
 MIN(WEATHER_DATE) AS START_DATE,
 MAX(WEATHER_DATE) AS END_DATE,
    SUM(CASE WHEN WEATHER_DATE IS NULL THEN 1 ELSE 0 END) AS weather_date_missing,
    SUM(CASE WHEN AIRPORT IS NULL THEN 1 ELSE 0 END) AS airport_missing,
    SUM(CASE WHEN TEMPERATURE IS NULL THEN 1 ELSE 0 END) AS temperature_missing,
    SUM(CASE WHEN VISIBILITY IS NULL THEN 1 ELSE 0 END) AS visibility_missing,
    SUM(CASE WHEN WINDSPEED IS NULL THEN 1 ELSE 0 END) AS windspeed_missing,
    SUM(CASE WHEN PRECIPITATION IS NULL THEN 1 ELSE 0 END) AS precipitation_missing,
    SUM(CASE WHEN WEATHER_FLAGS IS NULL THEN 1 ELSE 0 END) AS weather_flags_missing
FROM weather_cleaned;
"""

pd.read_sql_query(wq13_checkrows_cleaned,conn)

,ROW_COUNT,AIRPORT_COUNT,START_DATE,END_DATE,weather_date_missing,airport_missing,temperature_missing,visibility_missing,windspeed_missing,precipitation_missing,weather_flags_missing
0,2400,20,2025-01-01,2025-04-30,0,0,0,0,0,0,0


- Final `weather_cleaned` table recorded with 2400 rows and 20 airports with no missing values & within the preferred date ranges. 